# 05. Masked latent prediction and target updates

![Masked latent prediction](../images/05_masked_latent_prediction.svg)

**Learning goals:** build context and target masks, gather dense token subsets, predict target latents, verify stop-gradient, update a target encoder by exponential moving average, and run a minimal CPU training loop.

In [ ]:
import copy
import math
import random
import numpy as np
import torch
import torch.nn.functional as F

SEED = 29
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print(f'torch={torch.__version__}, device=cpu')

## 1. Structured synthetic tokens

Each sample is a 4 by 4 token grid. Sine, cosine, position, and sample amplitude form four input features. The structure makes hidden locations predictable from context without any external data download.

In [ ]:
B, grid_h, grid_w, input_width = 24, 4, 4, 4
N = grid_h * grid_w
position_1d = torch.linspace(0, 2*math.pi, N)
phase = torch.linspace(0.0, math.pi, B).unsqueeze(1)
amplitude = torch.linspace(0.7, 1.3, B).unsqueeze(1)
tokens = torch.stack([
    amplitude * torch.sin(position_1d + phase),
    amplitude * torch.cos(position_1d + phase),
    position_1d.expand(B, -1) / (2*math.pi),
    amplitude.expand(-1, N),
], dim=-1)
assert tokens.shape == (B, N, input_width)
assert torch.isfinite(tokens).all()
print('tokens:', tuple(tokens.shape))

## 2. Mask geometry and dense gather

Hide the center 2 by 2 block. Flattening uses `index = row*grid_width + column`, so targets are 5, 6, 9, and 10. Equal target counts let the batch use dense `(B,M,D)` tensors. `expand` broadcasts indices without allocating repeated copies.

In [ ]:
target_mask_grid = torch.zeros(grid_h, grid_w, dtype=torch.bool)
target_mask_grid[1:3, 1:3] = True
target_indices_1d = target_mask_grid.flatten().nonzero(as_tuple=False).squeeze(1)
context_indices_1d = (~target_mask_grid).flatten().nonzero(as_tuple=False).squeeze(1)
target_indices = target_indices_1d.expand(B, -1)
context_indices = context_indices_1d.expand(B, -1)

def gather_tokens(x, indices):
    expanded = indices.unsqueeze(-1).expand(-1, -1, x.shape[-1])
    return torch.gather(x, dim=1, index=expanded)

context_raw = gather_tokens(tokens, context_indices)
target_raw = gather_tokens(tokens, target_indices)
assert target_indices_1d.tolist() == [5, 6, 9, 10]
assert set(target_indices_1d.tolist()).isdisjoint(context_indices_1d.tolist())
assert context_raw.shape == (B, 12, input_width) and target_raw.shape == (B, 4, input_width)
print('context:', tuple(context_raw.shape), 'target:', tuple(target_raw.shape))

## 3. Online encoder, target encoder, and predictor

The same token MLP architecture is used for online and target encoders. The predictor combines a pooled context latent with the raw position features for each target location. This small baseline makes location information explicit.

In [ ]:
latent_width = 8
online_encoder = torch.nn.Sequential(
    torch.nn.Linear(input_width, 16), torch.nn.GELU(), torch.nn.Linear(16, latent_width))
target_encoder = copy.deepcopy(online_encoder)
for parameter in target_encoder.parameters():
    parameter.requires_grad_(False)

class LocationPredictor(torch.nn.Module):
    def __init__(self, latent_width, location_width=2):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(latent_width + location_width, 24),
            torch.nn.GELU(),
            torch.nn.Linear(24, latent_width))
    def forward(self, context_latents, target_locations):
        summary = context_latents.mean(dim=1, keepdim=True)
        summary = summary.expand(-1, target_locations.shape[1], -1)
        return self.net(torch.cat([summary, target_locations], dim=-1))

predictor = LocationPredictor(latent_width)
rows, cols = torch.meshgrid(torch.arange(grid_h), torch.arange(grid_w), indexing='ij')
locations = torch.stack([rows, cols], dim=-1).float().reshape(1, N, 2) / 3.0
target_locations = gather_tokens(locations.expand(B, -1, -1), target_indices)
assert target_locations.shape == (B, 4, 2)

## 4. Minimal autograd and optimizer primer

A model parameter is an adjustable tensor. During the forward pass, PyTorch records operations involving parameters whose `requires_grad` flag is true. A scalar **loss** summarizes prediction error. `zero_grad` clears gradients from the previous update, `loss.backward()` applies the chain rule and stores each parameter's gradient in `.grad`, and `optimizer.step()` uses those gradients to change parameters. Gradients accumulate across repeated `backward()` calls unless they are cleared. Stop-gradient with `torch.no_grad()` or `detach()` keeps a forward value but prevents the graph from continuing through that branch.

In [ ]:
# One scalar SGD step: forward, scalar loss, backward, then update.
toy_weight = torch.tensor(2.0, requires_grad=True)
toy_optimizer = torch.optim.SGD([toy_weight], lr=0.1)
toy_optimizer.zero_grad(set_to_none=True)
toy_prediction = toy_weight * 3.0
toy_loss = 0.5 * (toy_prediction - 5.0).square()
toy_loss.backward()
assert torch.isclose(toy_weight.grad, torch.tensor(3.0))
weight_before = toy_weight.detach().clone()
toy_optimizer.step()
assert toy_weight.item() < weight_before.item()
toy_optimizer.zero_grad(set_to_none=True)
assert toy_weight.grad is None
print(f'loss={toy_loss.item():.2f}, gradient=3.00, weight {weight_before.item():.2f} -> {toy_weight.item():.2f}')

## 5. Stop-gradient and EMA

The target branch runs under `torch.no_grad()`, so its forward values are available but no graph is retained. Target parameters are not in the optimizer. After the online update, EMA applies $\xi\leftarrow\tau\xi+(1-\tau)\theta$ in place with `mul_` and `add_`.

In [ ]:
@torch.no_grad()
def ema_update(online, target, tau):
    online_named = dict(online.named_parameters())
    target_named = dict(target.named_parameters())
    assert online_named.keys() == target_named.keys()
    for name, target_parameter in target_named.items():
        target_parameter.mul_(tau).add_(online_named[name], alpha=1.0 - tau)

def parameter_distance(a, b):
    return torch.sqrt(sum((pa.detach() - pb.detach()).square().sum()
                          for pa, pb in zip(a.parameters(), b.parameters()))).item()

assert parameter_distance(online_encoder, target_encoder) == 0.0
assert all(not p.requires_grad for p in target_encoder.parameters())
print('target initialized from online; gradients disabled')

## 6. One complete optimization step

Prediction and target shapes must match exactly before loss, or broadcasting can compare the wrong axes. `zero_grad(set_to_none=True)` avoids writing zeros into every old gradient buffer. EMA runs after `optimizer.step()` so it incorporates the latest online state.

In [ ]:
optimizer = torch.optim.Adam(list(online_encoder.parameters()) + list(predictor.parameters()), lr=3e-3)
tau = 0.98
optimizer.zero_grad(set_to_none=True)
context_latent = online_encoder(context_raw)
prediction = predictor(context_latent, target_locations)
with torch.no_grad():
    target_all = target_encoder(tokens)
    target_latent = gather_tokens(target_all, target_indices)
assert prediction.shape == target_latent.shape == (B, 4, latent_width)
loss = F.mse_loss(prediction, target_latent)
loss.backward()
assert any(p.grad is not None for p in online_encoder.parameters())
assert all(p.grad is None for p in target_encoder.parameters())
optimizer.step()
distance_before_ema = parameter_distance(online_encoder, target_encoder)
ema_update(online_encoder, target_encoder, tau)
distance_after_ema = parameter_distance(online_encoder, target_encoder)
assert 0 < distance_after_ema < distance_before_ema
print(f'loss={loss.item():.5f}, online-target distance {distance_before_ema:.4f} -> {distance_after_ema:.4f}')

## 7. Run a short deterministic training loop

This intentionally overfits a tiny fixed synthetic batch so the mechanics are easy to inspect. A real system would generate new samples and masks, schedule EMA momentum, and evaluate representation quality separately from prediction loss.

In [ ]:
loss_history = []
for step in range(80):
    optimizer.zero_grad(set_to_none=True)
    context_latent = online_encoder(context_raw)
    prediction = predictor(context_latent, target_locations)
    with torch.no_grad():
        target_latent = gather_tokens(target_encoder(tokens), target_indices)
    step_loss = F.mse_loss(prediction, target_latent)
    step_loss.backward()
    optimizer.step()
    ema_update(online_encoder, target_encoder, tau)
    loss_history.append(step_loss.item())

assert np.isfinite(loss_history).all()
assert all(p.grad is None for p in target_encoder.parameters())
print(f'first five mean={np.mean(loss_history[:5]):.5f}, last five mean={np.mean(loss_history[-5:]):.5f}')

## 8. Inspect invariants, not only loss

A small loss does not prove useful representations. Constant latents can also be easy to predict. At minimum, monitor feature variation and parameter finiteness, then evaluate representations on a separate task.

In [ ]:
with torch.no_grad():
    final_targets = target_encoder(tokens)
feature_std = final_targets.flatten(0, 1).std(dim=0)
assert torch.isfinite(feature_std).all()
assert all(torch.isfinite(p).all() for p in online_encoder.parameters())
print('target feature std range:', round(feature_std.min().item(), 4), 'to', round(feature_std.max().item(), 4))

# Frozen export has a stricter contract than a target branch used during training.
export_encoder = copy.deepcopy(target_encoder)
checkpoint = copy.deepcopy(target_encoder.state_dict())
export_encoder.load_state_dict(checkpoint, strict=True)
export_encoder.requires_grad_(False).eval()

@torch.inference_mode()
def export_frozen_features(model, batch):
    pre_norm_tokens = model(batch)
    if pre_norm_tokens.ndim != 3:
        raise ValueError("expected token features with shape [B, N, D]")
    features = pre_norm_tokens.mean(dim=1).float().cpu().numpy()
    if not np.isfinite(features).all():
        raise FloatingPointError("exported features are non-finite")
    return features

export_a = export_frozen_features(export_encoder, tokens)
export_b = export_frozen_features(export_encoder, tokens)
assert export_a.shape == (B, latent_width)
assert np.array_equal(export_a, export_b)
assert all(not parameter.requires_grad for parameter in export_encoder.parameters())
planned_final_step = 80
assert len(loss_history) == planned_final_step
print("deterministic frozen export:", export_a.shape, export_a.dtype)

## 9. Frozen inference export

Training and export need different contracts. `eval()` selects deterministic module behavior, `requires_grad_(False)` freezes parameters, and `torch.inference_mode()` disables autograd plus inference-time version tracking. Strict checkpoint loading rejects missing or unexpected parameters. Pool tokens while they are tensors, choose an interchange dtype, move to CPU, and only then convert to NumPy. In a planned comparison, export the final-step checkpoint for every condition. Downstream scores must not choose an earlier epoch, seed, or rerun.

## Efficiency, exercises, and final takeaways

**Efficiency:** encode all target tokens once and gather afterward. Keep the target branch under `no_grad`, use `expand` for index broadcasting, and use `inference_mode` for terminal frozen export. Verify masks are nonempty and disjoint. Never place target parameters in the gradient optimizer.

**Exercises:** change the block mask while preserving target count; vary `tau`; omit the location coordinates and inspect the predictions; compare `no_grad()` with `inference_mode()` for values that must later re-enter autograd; remove `eval()` from a model containing dropout and test repeatability.

**Takeaways:** mask geometry defines the prediction problem; context and target roles require careful indexing; stop-gradient creates optimization asymmetry; EMA supplies a slowly moving target; and frozen feature export needs a separate, explicit inference contract. Paired conditions should share named sequence, spatial, and mask streams while the intervention uses its own stream.

## Continue learning

[Previous notebook: 04](04_attention_and_positions.ipynb) | [Lecture](../lectures/05_masked_latent_prediction.md) | [Curriculum](../README.md) | [Next notebook: 06](06_representation_collapse.ipynb)